# Обучение моделей локализации источников в Google Colab

Этот notebook используется для настройки consistency-loss и финальных GPU-запусков на Facebook-датасете. Результаты и checkpoint сохраняются в Google Drive.

**Не нажимай `Run all`.** Длительные ячейки запускаются по одной и только в указанном порядке. После каждого обучения проверяй результат.

## Этап 1. Подключение Drive
Запускай эту ячейку первой после каждого нового Colab runtime. Все предыдущие результаты должны находиться в `MyDrive/diffusion-sources/reports`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/diffusion-sources'
REPOSITORY = 'https://github.com/1habibi/diffusion-sources-localization-.git'


## Этап 2. Проверка GPU
Если assertion завершился ошибкой, выбери `Runtime -> Change runtime type -> GPU` и перезапусти runtime. На CPU финальные запуски не выполнять.

In [ ]:
import torch

assert torch.cuda.is_available(), 'GPU недоступен: выбери GPU runtime'
print('PyTorch:', torch.__version__)
print('CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi


## Этап 3. Получение актуального кода
При первом запуске репозиторий клонируется. При повторном запуске выполняется `git pull --ff-only`. Все изменения должны быть заранее запушены в GitHub.

In [ ]:
from pathlib import Path

repo = Path('/content/diffusion-sources')
if not repo.exists():
    !git clone {REPOSITORY} /content/diffusion-sources
else:
    %cd /content/diffusion-sources
    !git pull --ff-only

%cd /content/diffusion-sources
!pip install -q networkx numpy matplotlib PyYAML scikit-learn streamlit tqdm torch-geometric
!pip install -q -e . --no-deps
!git rev-parse HEAD


## Этап 4. Проверка Facebook-датасета
Должны существовать четыре файла. Если проверка не проходит, загрузи локальный каталог `data/generated/facebook_main` в `MyDrive/diffusion-sources/data/facebook_main`.

In [ ]:
from pathlib import Path

DATA = Path(DRIVE_ROOT) / 'data/facebook_main'
required_files = ['graph.npz', 'train.npz', 'validation.npz', 'test.npz']
missing = [name for name in required_files if not (DATA / name).exists()]
assert not missing, f'Не найдены файлы: {missing}'

for name in required_files:
    path = DATA / name
    print(name, round(path.stat().st_size / 1024**2, 2), 'MiB')
print('Датасет готов:', DATA)


## Этап 5. Вспомогательные функции
Эта ячейка ничего не обучает. Она создает runtime-конфигурации, показывает итог запуска и сравнивает модели по validation. Запускай ее после каждого нового runtime.

In [ ]:
import json
from pathlib import Path
import yaml

def make_config(source_config, output_name, seed, *, consistency=None):
    config = yaml.safe_load(Path(source_config).read_text())
    config['data']['directory'] = f'{DRIVE_ROOT}/data/facebook_main'
    config['training']['device'] = 'cuda'
    config['training']['seed'] = int(seed)
    config['training']['resume'] = True
    if consistency is not None:
        config['loss']['lambda_consistency'] = float(consistency)
        config['experiment'] = f'facebook_joint_consistency_{consistency}'
    path = Path('/content') / f'{output_name}_{seed}.yaml'
    path.write_text(yaml.safe_dump(config, sort_keys=False))
    print('Создан конфиг:', path)
    return str(path)

def load_metrics(output_dir):
    return json.loads(Path(output_dir, 'metrics.json').read_text())

def show_result(output_dir, method='joint_estimated_k'):
    result_dir = Path(output_dir)
    metrics = load_metrics(result_dir)
    validation = metrics['metrics']['validation']
    test = metrics['prediction_metrics'][method]['all']
    print('Каталог:', result_dir)
    print('Лучшая эпоха:', metrics['best_epoch'])
    print('Остановка:', metrics['stopped_epoch'], metrics['stop_reason'])
    if metrics['model'] == 'node_only':
        print('Validation oracle-k F1:', round(validation['macro_f1'], 6))
        print('Validation thresholded F1:', round(metrics['validation_threshold_f1'], 6))
        print('Порог:', metrics['threshold'])
        print('Validation count accuracy: N/A (у Node-only нет count-head)')
    else:
        print('Validation F1:', round(validation['macro_f1'], 6))
        print('Validation count accuracy:', round(validation['count_accuracy'], 6))
    print('Test metrics:', json.dumps(test, indent=2))
    print('Время текущей сессии, ч:', round(metrics['training_seconds'] / 3600, 2))
    peak = metrics.get('peak_memory_bytes')
    print('Peak VRAM, GiB:', round(peak / 1024**3, 2) if peak else None)
    for name in ['last_checkpoint.pt', 'best_model.pt', 'history.json', 'history.csv', 'metrics.json', 'test_predictions.csv', 'config.yaml']:
        print(name, (result_dir / name).exists())

def validation_row(name, output_dir):
    metrics = load_metrics(output_dir)
    values = metrics['metrics']['validation']
    return {
        'name': name,
        'f1': values['macro_f1'],
        'count_accuracy': values['count_accuracy'],
        'count_mae': values['count_mae'],
    }


## Этап 6. Проверка уже готовых запусков seed 7026
Joint full, Node-only и Joint без consistency-loss уже обучены. Эта ячейка только читает результаты с Drive и ничего не переобучает. Если файл отсутствует, проверь путь на Drive.

In [ ]:
JOINT_FULL_7026 = f'{DRIVE_ROOT}/reports/joint_full/seed_7026'
NO_CONS_7026 = f'{DRIVE_ROOT}/reports/no_consistency/seed_7026'
NODE_7026 = f'{DRIVE_ROOT}/reports/node_only/seed_7026'

show_result(JOINT_FULL_7026)
show_result(NO_CONS_7026)
show_result(NODE_7026, 'node_thresholded')


# Контрольная точка A: tuning consistency-loss
Сейчас нужно проверить только `lambda_consistency=0.01` с seed 7026. Не запускай seed 7027/7028 до сравнения validation. Результаты test сохраняются автоматически, но коэффициент выбирается **только по validation**.

## Этап 7. Создание конфига consistency=0.01
Запусти ячейку один раз. Она не начинает обучение.

In [ ]:
CONS_001_CONFIG = make_config(
    'configs/train_facebook.yaml',
    'train_joint_consistency_001',
    7026,
    consistency=0.01,
)
CONS_001_7026 = f'{DRIVE_ROOT}/reports/tuning/joint_consistency_001/seed_7026'
print(Path(CONS_001_CONFIG).read_text())
print('Результат будет сохранен в:', CONS_001_7026)


## Этап 8. Обучение consistency=0.01 / seed 7026
Это длительная ячейка. Запускай ее отдельно и дождись `metrics.json`. При разрыве Colab повтори этапы 1-5 и эту же ячейку: `resume: true` продолжит checkpoint.

In [ ]:
!python scripts/train_model.py --config {CONS_001_CONFIG} --output {CONS_001_7026}


## Этап 9. Проверка tuning-запуска
Запускай только после успешного завершения этапа 8.

In [ ]:
show_result(CONS_001_7026)


## Этап 10. Сравнение коэффициентов по validation
Запускай после этапа 9. Победитель определяется максимальным validation F1. Если `0.01` и `0.0` практически равны, выбирай `0.0` как более простой вариант. Пришли эту таблицу перед запуском следующих seed.

In [ ]:
rows = [
    validation_row('consistency=0.1', JOINT_FULL_7026),
    validation_row('consistency=0.0', NO_CONS_7026),
    validation_row('consistency=0.01', CONS_001_7026),
]
for row in sorted(rows, key=lambda item: item['f1'], reverse=True):
    print(
        row['name'],
        'validation F1 =', round(row['f1'], 6),
        'count accuracy =', round(row['count_accuracy'], 6),
        'count MAE =', round(row['count_mae'], 6),
    )


# Контрольная точка B: финальные повторы
Ниже находятся заготовки следующих запусков. **Не запускай их до обсуждения таблицы этапа 10.** Сначала в ячейке выбора укажи победивший коэффициент.

Правило:
- если победил `0.01`, установи `BEST_CONSISTENCY = 0.01`;
- если `0.01` не лучше `0.0`, установи `BEST_CONSISTENCY = 0.0`;
- `0.1` больше не использовать без отдельного обоснования.

In [ ]:
# Выбрано по результатам этапа 10: validation F1 почти одинаков,
# но при 0.0 лучше count accuracy и ниже count MAE.
BEST_CONSISTENCY = 0.0
assert BEST_CONSISTENCY in (0.0, 0.01)


## Этап 11. Конфигурации финальной серии
Запускай после выбора `BEST_CONSISTENCY`. Эта ячейка только создает YAML-файлы.

In [ ]:
BEST_JOINT_7027 = make_config('configs/train_facebook.yaml', 'train_best_joint', 7027, consistency=BEST_CONSISTENCY)
BEST_JOINT_7028 = make_config('configs/train_facebook.yaml', 'train_best_joint', 7028, consistency=BEST_CONSISTENCY)
NO_CONS_7027_CONFIG = make_config('configs/train_facebook_no_consistency.yaml', 'train_no_consistency', 7027)
NO_CONS_7028_CONFIG = make_config('configs/train_facebook_no_consistency.yaml', 'train_no_consistency', 7028)
NODE_7027_CONFIG = make_config('configs/train_node_facebook.yaml', 'train_node', 7027)
NODE_7028_CONFIG = make_config('configs/train_node_facebook.yaml', 'train_node', 7028)
BEST_TAG = 'consistency_001' if BEST_CONSISTENCY == 0.01 else 'no_consistency'
print('Финальная Joint-конфигурация:', BEST_TAG)


## Этап 12. Best Joint / seed 7027
Длительный запуск. Выполняй этапы 12-13 только если `BEST_CONSISTENCY == 0.01`. Если выбран `0.0`, пропусти этапы 12-13 и переходи сразу к этапу 14: там находятся те же необходимые seed без consistency-loss.

In [ ]:
BEST_JOINT_OUT_7027 = f'{DRIVE_ROOT}/reports/final_joint/{BEST_TAG}/seed_7027'
!python scripts/train_model.py --config {BEST_JOINT_7027} --output {BEST_JOINT_OUT_7027}


In [ ]:
show_result(BEST_JOINT_OUT_7027)


## Этап 13. Best Joint / seed 7028
Запускай только после проверки seed 7027.

In [ ]:
BEST_JOINT_OUT_7028 = f'{DRIVE_ROOT}/reports/final_joint/{BEST_TAG}/seed_7028'
!python scripts/train_model.py --config {BEST_JOINT_7028} --output {BEST_JOINT_OUT_7028}


In [ ]:
show_result(BEST_JOINT_OUT_7028)


## Этап 14. Joint без consistency / seed 7027 и 7028
Эти запуски нужны как абляция. Если `BEST_CONSISTENCY=0.0`, они одновременно являются финальными Joint-запусками, поэтому этапы 12-13 дублировать не нужно. Каждый seed запускается отдельно.

In [ ]:
NO_CONS_OUT_7027 = f'{DRIVE_ROOT}/reports/no_consistency/seed_7027'
!python scripts/train_model.py --config {NO_CONS_7027_CONFIG} --output {NO_CONS_OUT_7027}


In [ ]:
show_result(NO_CONS_OUT_7027)


In [ ]:
NO_CONS_OUT_7028 = f'{DRIVE_ROOT}/reports/no_consistency/seed_7028'
!python scripts/train_model.py --config {NO_CONS_7028_CONFIG} --output {NO_CONS_OUT_7028}


In [ ]:
show_result(NO_CONS_OUT_7028)


## Этап 15. Node-only / seed 7027 и 7028
Node-only нужен как основной нейросетевой baseline. Каждый seed запускается отдельно.

In [ ]:
NODE_OUT_7027 = f'{DRIVE_ROOT}/reports/node_only/seed_7027'
!python scripts/train_node_model.py --config {NODE_7027_CONFIG} --output {NODE_OUT_7027}


In [ ]:
show_result(NODE_OUT_7027, 'node_thresholded')


In [ ]:
NODE_OUT_7028 = f'{DRIVE_ROOT}/reports/node_only/seed_7028'
!python scripts/train_node_model.py --config {NODE_7028_CONFIG} --output {NODE_OUT_7028}


In [ ]:
show_result(NODE_OUT_7028, 'node_thresholded')


# После финальной серии
Не запускай дополнительные модели без анализа. Проверь, что для каждого использованного каталога существуют `metrics.json`, `history.csv`, `history.json`, `best_model.pt`, `last_checkpoint.pt` и `test_predictions.csv`. После этого результаты нужно перенести в проект или обработать скриптами отчетности: средние значения, разброс и 95% confidence interval по трем seed, сравнение с baseline, robustness, IC-SI и Facebook-email.